# Темы 6-8: Множественная и Полиномиальная регрессия (ОТВЕТЫ)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

df = pd.merge(pd.read_csv('data/transactions.csv'), pd.read_csv('data/customers.csv'), on='customer_id')
df = pd.merge(df, pd.read_csv('data/products.csv'), on='product_id')
df['total_sum'] = df['price'] * df['quantity']
df['age'] = df['age'].fillna(df['age'].median())

### Задание 1

In [ ]:
X = df[['age','price','quantity']]
y = df['total_sum']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Задание 2

In [ ]:
model = LinearRegression().fit(X_train, y_train)
for name, coef in zip(X.columns, model.coef_):
    print(f'{name}: {coef:.4f}')
print(f'Intercept: {model.intercept_:.4f}')

### Задание 3

In [ ]:
preds = model.predict(X_test)
print(f'R²: {r2_score(y_test, preds):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, preds)):.2f}')
print(f'MAE: {mean_absolute_error(y_test, preds):.2f}')

### Задание 4

In [ ]:
df_enc = pd.get_dummies(df[['age','price','quantity','gender','location']], drop_first=True)
X2 = df_enc
y2 = df['total_sum']
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.2, random_state=42)
model2 = LinearRegression().fit(X2_tr, y2_tr)
print(f'R² с категориальными: {r2_score(y2_te, model2.predict(X2_te)):.4f}')

### Задание 5

In [ ]:
print(X.corr())

### Задание 6

In [ ]:
poly = PolynomialFeatures(degree=2)
X_poly_tr = poly.fit_transform(X_train)
X_poly_te = poly.transform(X_test)
print(f'Кол-во признаков: {X_poly_tr.shape[1]}')
poly_model = LinearRegression().fit(X_poly_tr, y_train)
print(f'R² (poly2): {r2_score(y_test, poly_model.predict(X_poly_te)):.4f}')

### Задание 7

In [ ]:
poly3 = PolynomialFeatures(degree=3)
X_p3_tr = poly3.fit_transform(X_train)
X_p3_te = poly3.transform(X_test)
model3 = LinearRegression().fit(X_p3_tr, y_train)
print(f'R² train: {r2_score(y_train, model3.predict(X_p3_tr)):.4f}')
print(f'R² test:  {r2_score(y_test, model3.predict(X_p3_te)):.4f}')
# Если R² train >> R² test → переобучение

### Задание 8

In [ ]:
plt.figure(figsize=(8,8))
plt.scatter(y_test, preds, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Реальные')
plt.ylabel('Предсказанные')
plt.title('Реальные vs Предсказанные')
plt.show()

### Задание 9

In [ ]:
importance = pd.DataFrame({'feature': X.columns, 'coef': model.coef_})
print(importance.sort_values('coef', ascending=False))

### Задание 10

In [ ]:
scaler = StandardScaler()
X_sc_tr = scaler.fit_transform(X_train)
X_sc_te = scaler.transform(X_test)
model_sc = LinearRegression().fit(X_sc_tr, y_train)
print(f'R² без scaler: {r2_score(y_test, model.predict(X_test)):.4f}')
print(f'R² с scaler:   {r2_score(y_test, model_sc.predict(X_sc_te)):.4f}')
# R² одинаковый, но коэффициенты теперь сопоставимы по масштабу

### Задание 11

In [ ]:
results = pd.DataFrame({
    'Модель': ['Simple (quantity)', 'Multiple (3 feat)', 'Poly deg=2', 'Poly deg=3'],
    'R² test': [
        r2_score(y_test, LinearRegression().fit(X_train[['quantity']], y_train).predict(X_test[['quantity']])),
        r2_score(y_test, model.predict(X_test)),
        r2_score(y_test, poly_model.predict(X_poly_te)),
        r2_score(y_test, model3.predict(X_p3_te))
    ]
})
print(results)

### Задание 12

In [ ]:
new = pd.DataFrame({'age': [28], 'price': [500], 'quantity': [2]})
print(f'Предсказание: {model.predict(new)[0]:.2f} руб.')
print(f'Ожидание: {500*2} руб.')